# Poetry: Dependency Management

## The Core Question

**How do you manage dependencies so your simulation environment is reproducible?**

Your CFD script needs `numpy` and `scipy`. You install "the latest" and it works. Three months later, a colleague installs "the latest" — which is now a *different* version — and your results don't match. "It works on my machine" is not a reproducible environment.

Poetry fixes this with a **lock file** that pins the *exact* versions of every dependency, so everyone gets the same environment.

## Why this library?

Poetry manages dependencies *and* creates a `poetry.lock` file that records the precise version of every package (and their transitive dependencies). Unlike a `requirements.txt` that lists ranges, the lock file is a **snapshot**: install it and you get byte-for-byte the same environment. For an engineer, that means: *reproducible environments = reproducible simulations.*

## The Workflow

```
  describe deps (pyproject.toml: numpy, scipy)
        |
        v
  poetry lock        -->  poetry.lock (exact pinned versions)
        |
        v
  poetry install     -->  everyone gets the SAME environment
        |
        v
  reproducible simulation runs, on every machine
```


In [ ]:
import subprocess, sys
out = subprocess.run([sys.executable, '-m', 'poetry', '--version'],
                     capture_output=True, text=True)
print(out.stdout.strip())

## 1. Describe the Project: `pyproject.toml`

Poetry reads a `pyproject.toml`. The `[project]` section lists the **dependencies as ranges** — "numpy at least 1.26", "scipy at least 1.11". A range says *what is acceptable*; the lock file (next) says *what we actually use*.

```toml
dependencies = [
    "numpy (>=1.26)",
    "scipy (>=1.11)",
]
```

This is the *intent*. It's human-readable and flexible — you can bump the floor later without breaking the lock.


In [ ]:
# pyproject.toml
[project]
name = "my-cfd-sim"
version = "0.1.0"
description = "CFD simulation environment"
authors = [{"name" = "tschuller"}]
requires-python = "^3.11"
dependencies = [
    "numpy (>=1.26)",
    "scipy (>=1.11)",
]

[tool.poetry]
packages = [{include = "my_cfd_sim"}]

## 2. Lock the Dependencies: `poetry lock`

`poetry lock` resolves the ranges into **exact versions** and writes them to `poetry.lock`. This is the critical step: it turns "numpy >= 1.26" into "numpy **2.4.6**" (the specific version that satisfies the constraint *today*).

The lock file also records every **transitive** dependency and a content hash, so any change to `pyproject.toml` invalidates it.


In [ ]:
import subprocess, sys, os

# Run poetry lock in the demo project directory
demo = os.path.join(os.getcwd(), '_poetry_demo')
env = dict(os.environ, POETRY_VIRTUALENVS_CREATE='false')
result = subprocess.run([sys.executable, '-m', 'poetry', 'lock'],
                        cwd=demo, capture_output=True, text=True, env=env)
print(result.stdout)
if result.stderr:
    print(result.stderr)

# Show what got pinned
import re
lock = os.path.join(demo, 'poetry.lock')
if os.path.exists(lock):
    txt = open(lock, encoding='utf-8').read()
    names = re.findall(r'name = "([a-z0-9_\-]+)"', txt)
    vers = re.findall(r'version = "([0-9][^"]+)"', txt)
    print('Pinned in poetry.lock:')
    for n, v in zip(names, vers):
        print(f'  {n} == {v}')
    print(f'  ({len(txt.splitlines())} lines total)')

### Reading the output

```
Updating dependencies
Resolving dependencies...

Writing lock file

Pinned in poetry.lock:
  numpy == 2.4.6
  scipy == 1.17.1
  (167 lines total)
```

The lock file now says: *this project uses numpy 2.4.6 and scipy 1.17.1, exactly.* The 167 lines include hashes and metadata that let Poetry verify the environment is intact.

**Engineering takeaway:** the lock file is the **reproducibility contract**. Commit it to version control. When a colleague runs `poetry install`, they get *these exact versions* — not "whatever is latest." That's the difference between "my results" and "our results."


## 3. Install the Environment: `poetry install`

`poetry install` reads `poetry.lock` and installs the pinned versions into an isolated virtual environment. Because it installs from the lock file (not the ranges), **every machine gets the same environment**.

```
$ poetry install
Installing dependencies from lock file

Package operations: 2 installs

  • Installing numpy 2.4.6
  • Installing scipy 1.17.1
```

No version guessing, no "works on my machine."


In [ ]:
import subprocess, sys, os

demo = os.path.join(os.getcwd(), '_poetry_demo')
env = dict(os.environ, POETRY_VIRTUALENVS_CREATE='false')
result = subprocess.run([sys.executable, '-m', 'poetry', 'install', '--no-root'],
                        cwd=demo, capture_output=True, text=True, env=env)
print(result.stdout)
if result.stderr:
    print(result.stderr)

### What does this mean?

`poetry install` reproduced the exact environment from the lock file. If you delete the virtual environment and run it again, you get the *same* numpy and scipy versions — every time, on every machine.

**Engineering takeaway:** for a simulation whose results depend on library behavior (a solver's convergence, a linear-algebra routine's rounding), a pinned environment is not a nice-to-have — it's a **correctness requirement**. Poetry gives you that with one lock file and one command.


## 4. Exercises

1. **Add a dependency.** Run `poetry add matplotlib` in the demo project. Re-run `poetry lock`. What new entries appear in `poetry.lock`? (Hint: matplotlib pulls in several transitive dependencies.)

2. **Break the lock on purpose.** Edit `pyproject.toml` to change the numpy floor to `>=2.0`. Run `poetry lock`. Did the pinned numpy version change? What does that tell you about when the lock file is regenerated?

3. **Reproducibility test.** Note the pinned versions. Delete the virtual environment (`poetry env remove --all`), then `poetry install` again. Are the versions identical? (They should be.)

4. **Compare to requirements.txt.** Create a `requirements.txt` with `numpy` and `scipy` (no versions). Install it in a fresh venv. What versions do you get? Why is that *less* reproducible than the Poetry lock file?
